# Multi-Head Attention (MHA) 讲解与实现

这个 notebook 用一个可运行的 PyTorch 示例来解释 MHA 的核心思路：

1. 先把输入映射成 `Q`、`K`、`V`
2. 按头数切分成多个 attention head
3. 每个 head 独立做 scaled dot-product attention
4. 拼接所有 head，再通过输出线性层投影回原维度

常见公式如下：

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

多头注意力会把表示空间拆成多个子空间，让模型从不同角度关注序列中的信息。

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
torch.set_printoptions(precision=4, sci_mode=False)

## 从零实现一个简化版 MHA

下面这个实现保留了 MHA 的关键结构：

- `q_proj / k_proj / v_proj`：把输入投影成查询、键、值
- `reshape_heads`：把最后一维拆成 `num_heads x head_dim`
- attention 分数：`Q @ K^T / sqrt(head_dim)`
- `softmax` 后与 `V` 相乘得到每个 head 的输出
- 最后拼接并过 `out_proj`

为了便于教学，这里暂时不加 mask，也不处理 dropout。

In [ ]:
class SimpleMultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model 必须能被 num_heads 整除")

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def reshape_heads(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        x = x.view(batch_size, seq_len, self.num_heads, self.head_dim)
        return x.transpose(1, 2)

    def forward(self, x: torch.Tensor):
        q = self.reshape_heads(self.q_proj(x))
        k = self.reshape_heads(self.k_proj(x))
        v = self.reshape_heads(self.v_proj(x))

        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)
        attn_weights = F.softmax(scores, dim=-1)

        context = attn_weights @ v
        context = context.transpose(1, 2).contiguous()
        batch_size, seq_len, _, _ = context.shape
        context = context.view(batch_size, seq_len, self.d_model)

        output = self.out_proj(context)
        return output, attn_weights

## 跑一个最小示例

这里构造一个 `batch=2, seq_len=4, d_model=8` 的输入，并观察：

- 输出张量 `output` 的形状
- 注意力权重 `attn_weights` 的形状

如果 `num_heads=2`，那么每个 head 的维度就是 `4`。

In [ ]:
batch_size = 2
seq_len = 4
d_model = 8
num_heads = 2

x = torch.randn(batch_size, seq_len, d_model)
mha = SimpleMultiHeadAttention(d_model=d_model, num_heads=num_heads)

output, attn_weights = mha(x)

print("input shape :", x.shape)
print("output shape:", output.shape)
print("attn shape  :", attn_weights.shape)
print()
print("第 1 个样本、第 1 个 head 的注意力矩阵：")
print(attn_weights[0, 0])

## 代码里的维度流转

假设输入 `x` 的形状是 `(B, T, D)`：

- 投影后 `q/k/v` 仍然是 `(B, T, D)`
- 切分 head 后变成 `(B, H, T, D/H)`
- `q @ k^T` 后得到注意力分数 `(B, H, T, T)`
- 和 `v` 相乘后得到 `(B, H, T, D/H)`
- 最后拼接回 `(B, T, D)`

因此，MHA 的重点其实是“并行地在多个子空间做 attention”。

## 和 PyTorch 官方模块对照

`torch.nn.MultiheadAttention` 是工业实现，支持更多特性，比如：

- `attn_mask`
- `key_padding_mask`
- dropout
- 更高效的内部实现

下面给一个最小调用例子，帮助把“教学版实现”和“实际工程用法”对应起来。

In [ ]:
official_mha = nn.MultiheadAttention(
    embed_dim=d_model,
    num_heads=num_heads,
    batch_first=True,
)

official_output, official_weights = official_mha(x, x, x)

print("official output shape:", official_output.shape)
print("official weights shape:", official_weights.shape)

## 小结

MHA 的价值在于：

- 让模型同时关注不同位置关系
- 让不同 head 学到不同的交互模式
- 通过并行计算获得较强的表达能力

如果你下一步想继续扩展，可以尝试自己加上：

- causal mask
- padding mask
- dropout
- cross attention（`Q` 来自 decoder，`K/V` 来自 encoder）